# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [2]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\pembe\AppData\Local\Temp\ipykernel_25744\4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [4]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [5]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [6]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [7]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata enables source attribution, so the model can cite which document and page each answer came from, making responses verifiable and trustworthy. It also allows retrieval to be filtered or scoped by document type, date, or other fields, so the vector search returns contextually appropriate chunks rather than mixing unrelated sources.

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [8]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [9]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

Larger chunks preserve more context but make retrieval less precise; smaller chunks are more targeted but may miss surrounding information. More overlap reduces lost context at boundaries but creates more chunks, more duplicate results, and higher cost.

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [10]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [11]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [12]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

A similarity score indicates how closely a chunk's embedding matches the query's embedding in vector space, however, it does not prove the chunk is correct, complete or actually answers the question.

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [13]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [14]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [15]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [16]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that your cat may need veterinary attention can include changes in behavior or body signs such as:

- Pain or anxiety, especially if subtle or new [Source 1]
- Reduced grooming or increased grooming [Source 1][Source 2]
- Changes in appetite, vomiting, vomiting hairballs, or diarrhea [Source 3]
- Polyuria/polydipsia, increased nocturnal activity, or vocalization [Source 3]
- Changes in normal habits or activity [Source 3]
- Signs of fear or stress such as cowering, crouching, hiding, freezing, frantic fleeing, flattened ears, dilated pupils, tense body posture, or hissing/growling/yowling [Source 4]
- House-soiling or aggression toward people or other animals [Source 2]

If you’re seeing any of these, it’s a good idea to contact a veterinarian for an evaluation.

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 8, 'start_index': 0, 'score': 0.5962143181380077}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 8, 'start_ind

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [17]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guidelines recommend individualized preventive care based on a cat’s life stage and risk factors, with at least annual veterinary exams for all cats and more frequent visits for senior cats or those with chronic conditions [Source 4]. Preventive healthcare should include parasite prevention, especially broad-spectrum products for most pet cats, and extra attention for cats with outdoor access or time away from home [Source 1]. The guidelines also mention client education topics such as sterilization, claw care, identification/microchipping, and disaster preparedness [Source 4].
Question: What symptoms should make me call a veterinarian?
You should contact a veterinarian if your cat has changes in appetite, increased urination or thirst, vomiting, vomiting hairballs, diarrhea, or ongoing GI signs, since these can help guide diagnostic testing and may indicate disease [Source 1]. Also watch for increased nocturnal activity or vo

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

Yes, for the first three questions the retrieved context would likely be relevant, because the queries use vocabulary (preventive care, symptoms, veterinarian, feeding) that closely matches the kind of language in a cat health guideline PDF i.e. the embeddings land near the right chunks. The fourth question ("Can my cat help me file my taxes?") would retrieve poorly matched chunks with low scores, since nothing in the PDF is about taxes; the model should then correctly respond that it doesn't have enough information. This illustrates that dense vector retrieval works well when the query vocabulary overlaps with the document vocabulary, but degrades gracefully for out-of-domain questions rather than hallucinating an answer.

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [18]:
# Activity: Tune Retrieval Quality
# Experiment: smaller chunk size (500) vs baseline (1000)
# Variable changed: chunk_size and chunk_overlap only — one change at a time.

ACTIVITY_QUERY = "What vaccinations are recommended for cats?"
ACTIVITY_K = 4

#── Baseline ──────────────────────────────────────────────────────────────────

print("=" * 80)
print("BASELINE chunk_size=1000 chunk_overlap=200 k=4")
print("=" * 80)
baseline_results = display_retrieval_results(ACTIVITY_QUERY, k=ACTIVITY_K)
baseline_scores = [round(score, 3) for _, score in baseline_results]
print("Baseline scores:", baseline_scores)

baseline_answer = answer_question(ACTIVITY_QUERY, k=ACTIVITY_K)["answer"]
print("\nBaseline answer:\n", baseline_answer)

BASELINE chunk_size=1000 chunk_overlap=200 k=4
Source 1 | score=0.681 | page=15 | start_index=3894
The Task Force supports the2020 AAHA/AAFP Feline Vaccination Guidelines 7 and the World Small Animal V eterinary Association’sr e c - ommendation that veterinarians should vaccinate every animal with core vaccines and give non-core vaccines no more frequently than is deemed necessary based on risk exposure.126 Revaccination against FPV , FHV-1, and
--------------------------------------------------------------------------------
Source 2 | score=0.623 | page=15 | start_index=4721
nually for individual cats at high risk. V eterinarians have considerable ability to use biologics in a discretionary manner but also should be aware of any state- or provincial-speciﬁc restrictions in their veterinary practice act relating to implementation, especially in regard to rabies. Detailed information regarding the role of vaccination as 
------------------------------------------------------------------

In [19]:
# ── Experiment A: smaller chunks ──────────────────────────────────────────────
# Rebuild splitter and vector store with new chunk settings.
exp_chunk_size = 500
exp_chunk_overlap = 100

exp_splitter = RecursiveCharacterTextSplitter(
chunk_size=exp_chunk_size,
chunk_overlap=exp_chunk_overlap,
add_start_index=True,
)

exp_splits = exp_splitter.split_documents(pages)
print(f"Experiment: split {len(pages)} pages into {len(exp_splits)} chunks "
f"(was {len(splits)} with baseline settings).")

exp_vector_store = QdrantVectorStore.from_documents(
documents=exp_splits,
embedding=embeddings,
location=":memory:",
collection_name="cat_health_guidelines_exp",
force_recreate=True,
)
print(f"Built experiment vector store chunk_size={exp_chunk_size} chunk_overlap={exp_chunk_overlap}")

Experiment: split 22 pages into 263 chunks (was 135 with baseline settings).
Built experiment vector store chunk_size=500 chunk_overlap=100


In [20]:
# ── Compare retrieval results ─────────────────────────────────────────────────
def display_retrieval_results_from_store(store, query: str, k: int) -> list[tuple]:
    results = store.similarity_search_with_score(query, k=k)
    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("", " ")
        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)
    return results

print("=" * 80)
print(f"EXPERIMENT  chunk_size={exp_chunk_size}  chunk_overlap={exp_chunk_overlap}  k={ACTIVITY_K}")
print("=" * 80)
exp_results = display_retrieval_results_from_store(exp_vector_store, ACTIVITY_QUERY, k=ACTIVITY_K)
exp_scores = [round(score, 3) for _, score in exp_results]
print("Experiment scores:", exp_scores)

EXPERIMENT  chunk_size=500  chunk_overlap=100  k=4
Source 1 | score=0.680 | page=15 | start_index=4028
 o m m e n d a t i o n   t h a t   v e t e r i n a r i a n s   s h o u l d   v a c c i n a t e   e v e r y   a n i m a l   w i t h 
 c o r e   v a c c i n e s   a n d   g i v e   n o n - c o r e   v a c c i n e s   n o   m o r e   f r e q u e n t l y   t h a n   i s 
 d e e m e d   n e c e s s a r y   b a s e d   o n   r i s k   e x p o s u r e . 1 2 6   R e v a c c i n a t i o n   a g a i n s t   F P V   , 
 F H V - 1 ,   a n d   F C V   a t   6   m o n t h s   o f   a g e   t o   p o t e n t i a l l y   r e d u c e   t h e   w i n d o w 
 o f   s u s c e p t i b i l i t y   i n   k i t t e n s   w i t h   m a t e   r n a l l y   d e r i v e d   a n t i b o d i e s   t o w a r d 
 t h e   e 
--------------------------------------------------------------------------------
Source 2 | score=0.641 | page=15 | start_index=4433
 7   F e l i n e 
 l e u k e m i a   v i r u s   ( F e L V )  

In [21]:
# ── Compare generated answers ─────────────────────────────────────────────────
def answer_question_from_store(store, question: str, k: int) -> str:
    scored_docs = store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    return rag_chain.invoke({"context": context, "question": question})

exp_answer = answer_question_from_store(exp_vector_store, ACTIVITY_QUERY, k=ACTIVITY_K)

print("BASELINE answer:")
print(baseline_answer)
print()
print("EXPERIMENT answer:")
print(exp_answer)

print()
print("Score comparison:")
print(f" Baseline scores : {baseline_scores}")
print(f" Experiment scores : {exp_scores}")
print(f" Baseline avg : {sum(baseline_scores)/len(baseline_scores):.3f}")
print(f" Experiment avg : {sum(exp_scores)/len(exp_scores):.3f}")

BASELINE answer:
The context says cats should receive **core vaccines**: **rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV)**. **Non-core vaccines** may be given based on a cat’s exposure and risk factors [Source 4].

It also notes **feline leukemia virus (FeLV) vaccination** is considered **core for kittens and young cats**, especially those with regular or high-risk exposure [Source 1].

For a cat-specific vaccination plan, it’s best to discuss with a veterinarian [Source 1].

EXPERIMENT answer:
The provided guideline says veterinarians should give **core vaccines to every cat** and **non-core vaccines only as needed based on risk exposure** [Source 1]. It specifically mentions **FPV, FHV-1, and FCV** as core vaccines with a recommendation for **revaccination at 6 months of age** in kittens to reduce the window of susceptibility [Source 1]. It also says **FeLV vaccination is considered core for kittens and young cats**, e

### 🏗️ Activity Notes

- Setting changed: chunk_size 1000 → 500, chunk_overlap 200 → 100
- Before: 4 chunks retrieved with scores [0.681, 0.623, 0.623, 0.621], avg 0.637. 3 of 4 chunks came from page 15 (the vaccination page), with adjacent start_indexes (3160, 3894, 4721) - indicating the baseline already retrieved topically focused chunks for this specific query, not a mix of unrelated topics. The fourth chunk came from page 19.
- After: 263 chunks vs 135 in baseline. Each chunk is half the size, so each embedding covers a narrower slice of text.
- Did retrieval improve? Why or why not? : The baseline already landed on the right section (scores 0.62–0.68, all from the vaccination pages), so there was little to fix. Smaller chunks may push scores marginally higher, but risk splitting a multi-sentence vaccination recommendation across chunk boundaries, making the generated answer less complete. An improvement could be rewording the query to match the document's vocabulary more closely.